In [1]:
import torch
import torch.nn as nn
import os
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import random
import torch.optim as optim
import dotenv
print("Done")

Done


In [2]:
dotenv.load_dotenv()

path = os.getenv("MAIN_PATH")

In [3]:
test_path = os.path.join(path,"Test","Test")
train_path = os.path.join(path,"Train","Train")
op = "output"
#data imbalance check
for i in os.listdir(train_path):
    print(len(os.listdir(os.path.join(train_path,i))))



3066
2975
2979
2962
2926
3133
2921
2711
3371
2959


In [10]:
#making train path
os.makedirs(os.path.join(op,"Train"),exist_ok=True)
os.makedirs(os.path.join(op,"Test"),exist_ok=True)

In [14]:
import random

# Define 'op' as it's used in this context for the output directory
train_output_path = os.path.join(path, "Train","Train")



print("Checking for processed training data to randomly remove half...")

if os.path.exists(train_output_path):
    for class_name in os.listdir(train_output_path):
        class_dir = os.path.join(train_output_path, class_name)
        if os.path.isdir(class_dir):
            all_pt_files = [f for f in os.listdir(class_dir) if f.endswith('.bin')]
            random.shuffle(all_pt_files)

            num_files_to_remove = len(all_pt_files) // 2
            files_to_remove = all_pt_files[:num_files_to_remove]

            for filename in files_to_remove:
                file_path_to_delete = os.path.join(class_dir, filename)
                os.remove(file_path_to_delete)
            print(f"Removed {num_files_to_remove} files from class '{class_name}' in training data.")
else:
    print(f"Training output directory '{train_output_path}' does not exist yet. No processed files to remove.")

Checking for processed training data to randomly remove half...
Removed 3065 files from class '3' in training data.
Removed 2974 files from class '9' in training data.
Removed 2979 files from class '2' in training data.
Removed 2961 files from class '0' in training data.
Removed 2925 files from class '8' in training data.
Removed 3132 files from class '7' in training data.
Removed 2921 files from class '4' in training data.
Removed 2710 files from class '5' in training data.
Removed 3371 files from class '1' in training data.
Removed 2959 files from class '6' in training data.


In [15]:
def process_data(path,sub_path):
    for i in os.listdir(path):
      cat_path = os.path.join(path,i)
      os.makedirs(os.path.join(op, sub_path, i),exist_ok=True)

      for j in (os.listdir(cat_path)):
        check_size = np.fromfile(os.path.join(cat_path,j),dtype = np.uint8)
        check_size = check_size.reshape(-1,5)
        # print(check_size.shape)
        X = check_size[:,0]
        Y = check_size[:,1]
        polarity =(check_size[:,2] & 0x80 ) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)

            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)

        pt_filename = j.replace(".bin", ".pt")
        save_sub_path = os.path.join(op, sub_path, i, pt_filename)

        torch.save(torch_set, save_sub_path)
        print(f"Saved file {pt_filename} with shape {torch_set.shape}")

In [ ]:
process_data(train_path,"Train")

In [19]:
for i in os.listdir(test_path):
    print(len(os.listdir(os.path.join(test_path,i))))


1010
1009
1032
980
974
1028
982
892
1135
958


In [15]:
import shutil
import random
import numpy as np


# Clean up existing output directories to ensure a fresh split
shutil.rmtree(os.path.join(op, "Test"), ignore_errors=True)
shutil.rmtree(os.path.join(op, "Validate"), ignore_errors=True)

# Define raw test path (assuming 'path' variable is still in scope from previous cells)
test_path_raw = test_path

# Create output directories
os.makedirs(os.path.join(op, "Test"), exist_ok=True)
os.makedirs(os.path.join(op, "Validate"), exist_ok=True)

for class_name in os.listdir(test_path_raw):
    class_raw_path = os.path.join(test_path_raw, class_name)

    # Ensure output directories exist for this class
    os.makedirs(os.path.join(op, "Test", class_name), exist_ok=True)
    os.makedirs(os.path.join(op, "Validate", class_name), exist_ok=True)

    all_files_in_class = [f for f in os.listdir(class_raw_path) if f.endswith('.bin')]
    random.shuffle(all_files_in_class) # Shuffle to ensure random split

    # Split into two halves
    half_point = len(all_files_in_class) // 2
    test_split_files = all_files_in_class[:half_point]
    validate_split_files = all_files_in_class[half_point:]

    print(f"Class {class_name}: {len(test_split_files)} files for Test, {len(validate_split_files)} files for Validate")

    # Process and save files for the 'Test' split
    for filename in test_split_files:
        full_file_path = os.path.join(class_raw_path, filename)
        check_size = np.fromfile(full_file_path, dtype=np.uint8)
        check_size = check_size.reshape(-1, 5)
        X = check_size[:, 0]
        Y = check_size[:, 1]
        polarity = (check_size[:, 2] & 0x80) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)
            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)
        pt_filename = filename.replace(".bin", ".pt")

        target_dir = os.path.join(op, "Test", class_name)
        save_path = os.path.join(target_dir, pt_filename)
        torch.save(torch_set, save_path)

    # Process and save files for the 'Validate' split
    for filename in validate_split_files:
        full_file_path = os.path.join(class_raw_path, filename)
        check_size = np.fromfile(full_file_path, dtype=np.uint8)
        check_size = check_size.reshape(-1, 5)
        X = check_size[:, 0]
        Y = check_size[:, 1]
        polarity = (check_size[:, 2] & 0x80) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)
            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)
        pt_filename = filename.replace(".bin", ".pt")

        target_dir = os.path.join(op, "Validate", class_name)
        save_path = os.path.join(target_dir, pt_filename)
        torch.save(torch_set, save_path)

Class 3: 505 files for Test, 505 files for Validate
Class 9: 504 files for Test, 505 files for Validate
Class 2: 516 files for Test, 516 files for Validate
Class 0: 490 files for Test, 490 files for Validate
Class 8: 487 files for Test, 487 files for Validate
Class 7: 514 files for Test, 514 files for Validate
Class 4: 491 files for Test, 491 files for Validate
Class 5: 446 files for Test, 446 files for Validate
Class 1: 567 files for Test, 568 files for Validate
Class 6: 479 files for Test, 479 files for Validate


In [16]:
from torchvision.datasets import DatasetFolder

# Define a dummy loader for .pt files
def pt_loader(path):
    return torch.load(path)

# Create DatasetFolder instances
train_dataset = DatasetFolder(
    root=os.path.join(op, "Train"),
    loader=pt_loader,
    extensions=('.pt',)
)

test_dataset = DatasetFolder(
    root=os.path.join(op, "Test"),
    loader=pt_loader,
    extensions=('.pt',)
)

val_dataset = DatasetFolder(
    root=os.path.join(op, "Validate"),
    loader=pt_loader,
    extensions=('.pt',)
)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of samples in training : {len(train_dataset)}")
print(f"Number of samples in test : {len(test_dataset)}")
print(f"Number of samples in validation : {len(val_dataset)}")
# print(next(iter(train_loader)))


Number of samples in training : 30003
Number of samples in test : 4999
Number of samples in validation : 5001


In [17]:
class CNN(nn.Module):
    def __init__(self,ip):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv3d(in_channels=ip, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2),
            nn.Conv3d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2)
        )

        zeros = torch.zeros(1,ip,30,34,34)
        op = self.conv_layer(zeros)
        self.flatten_size = op.shape[1]*op.shape[2]*op.shape[3]* op.shape[4]
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.classifier(x)
        return x

In [18]:
train_loss = np.array([])
test_loss = np.array([])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ip_channel = 2
model = CNN(ip_channel).to(device)
epochs = 10
learning_rate = 0.0001
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(),lr = learning_rate,weight_decay=1e-4)

In [19]:
device

device(type='cuda')

In [26]:
for epoch in range(epochs):
    train_loss_perCycle=0
    test_loss_perCycle=0
    model.train()
    for X_train,y_train in train_loader:
        X_train,y_train = X_train.to(device), y_train.long().to(device)
        y_pred = model(X_train)
        loss = loss_func(y_pred, y_train)
        optimizer.zero_grad()
        #Backpropagation
        loss.backward()

        #update
        optimizer.step()
        train_loss_perCycle += loss.item()
    avg_train_loss = train_loss_perCycle/len(train_loader)
    train_loss = np.append(train_loss,avg_train_loss)

    model.eval()
    with torch.no_grad():
        for x_test,y_test in test_loader:
            x_test,y_test = x_test.to(device), y_test.long().to(device)
            y_test_pred = model(x_test)
            loss = loss_func(y_test_pred,y_test)
            test_loss_perCycle += loss.item()
        avg_test_loss = test_loss_perCycle/len(test_loader)
        test_loss = np.append(test_loss,avg_test_loss)
    print(f"Train Loss for round {epoch + 1} is {avg_train_loss} | Test Loss for round {epoch + 1} is {avg_test_loss}")

Train Loss for round 1 is 0.6556739030298648 | Test Loss for round 1 is 0.22229831057447422
Train Loss for round 2 is 0.2147138815984797 | Test Loss for round 2 is 0.12386912426946661
Train Loss for round 3 is 0.14556186185129036 | Test Loss for round 3 is 0.09043623702129043
Train Loss for round 4 is 0.11048129626087097 | Test Loss for round 4 is 0.07920622311626808
Train Loss for round 5 is 0.09037797802936103 | Test Loss for round 5 is 0.07405742496116488
Train Loss for round 6 is 0.07518686963987947 | Test Loss for round 6 is 0.06915544805483881
Train Loss for round 7 is 0.06594570715730982 | Test Loss for round 7 is 0.062564236698068
Train Loss for round 8 is 0.05626534578154114 | Test Loss for round 8 is 0.0530853343219791
Train Loss for round 9 is 0.050700898360171075 | Test Loss for round 9 is 0.06440814705477155
Train Loss for round 10 is 0.0421206108632007 | Test Loss for round 10 is 0.04902783717187947


In [27]:
predicted = np.array([])
actual = np.array([])
cnt = 0
with torch.no_grad():
    for X_val,y_val in val_loader:
        X_val,y_val = X_val.to(device),y_val.to(device)
        y_val_pred = model(X_val).squeeze()
        y_val_pred =  torch.argmax(y_val_pred,dim=1).cpu().numpy()
        print(f"y_val_pred for {cnt} is:",y_val_pred)
        predicted = np.concatenate([y_val_pred,predicted],axis=0)
        actual = np.concatenate([y_val.cpu().numpy(),actual],axis=0)
        cnt += 1

y_val_pred for 0 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 1 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 8 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 2 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 3 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 4 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 5 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
y_val_pred for 6 is: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

In [29]:
labels = list(range(len(val_dataset.classes)))
print("val classes:", val_dataset.classes)
print("val class_to_idx:", val_dataset.class_to_idx)

val classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
val class_to_idx: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}


In [30]:
from sklearn.metrics import confusion_matrix,classification_report

In [31]:
from sklearn.metrics import accuracy_score

print("VALIDATION RESULTS:")
model.eval()
print(actual.shape, predicted.shape)

matrix = confusion_matrix(actual, predicted, labels=labels)
report = classification_report(
    actual,
    predicted,
    labels=labels,
    target_names=val_dataset.classes,
    zero_division=0,
)
print("Accuracy:", accuracy_score(actual, predicted))
print("Confusion Matrix:\n", matrix)
print("Classification Report:\n", report)

VALIDATION RESULTS:
(5001,) (5001,)
Accuracy: 0.9838032393521295
Confusion Matrix:
 [[488   0   0   0   1   0   0   0   1   0]
 [  0 566   1   0   0   0   1   0   0   0]
 [  3   0 509   0   1   0   0   1   2   0]
 [  0   0   3 495   0   4   0   2   1   0]
 [  0   0   0   0 489   0   0   0   0   2]
 [  1   0   0   2   0 441   1   0   0   1]
 [  1   2   0   0   0   1 475   0   0   0]
 [  1   2   4   0   2   0   0 501   2   2]
 [  7   0   4   2   1   0   1   2 466   4]
 [  2   0   0   0   9   1   0   2   1 490]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98       490
           1       0.99      1.00      0.99       568
           2       0.98      0.99      0.98       516
           3       0.99      0.98      0.99       505
           4       0.97      1.00      0.98       491
           5       0.99      0.99      0.99       446
           6       0.99      0.99      0.99       479
           7       0.99     

In [32]:
model.eval()
pred_batches = []
actual_batches = []
with torch.inference_mode():
    for x_train, y_train in train_loader:
        x_train, y_train = x_train.to(device), y_train.to(device)
        logits = model(x_train)
        y_pred = logits.argmax(dim=1).cpu().numpy()
        pred_batches.append(y_pred)
        actual_batches.append(y_train.cpu().numpy())

pred_train = np.concatenate(pred_batches, axis=0)
actual_train = np.concatenate(actual_batches, axis=0)

In [33]:
print("TRAIN")
matrix_train = confusion_matrix(actual_train,pred_train,labels=[0,1,2,3,4,5])
report_train = classification_report(actual_train,pred_train,target_names=[str(x) for x in range(10)])
print("Confusion Matrix:\n",matrix_train)
print("Classification Report:\n",report_train)

TRAIN
Confusion Matrix:
 [[2961    0    0    0    0    0]
 [   0 3364    2    0    1    0]
 [   0    2 2966    1    2    0]
 [   1    1    3 3046    1    1]
 [   1    4    0    0 2915    0]
 [   3    0    0    1    1 2700]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00      2962
           1       1.00      1.00      1.00      3371
           2       1.00      1.00      1.00      2979
           3       1.00      0.99      1.00      3066
           4       0.99      1.00      0.99      2921
           5       1.00      1.00      1.00      2711
           6       1.00      1.00      1.00      2959
           7       0.99      1.00      1.00      3133
           8       1.00      0.98      0.99      2926
           9       0.99      0.99      0.99      2975

    accuracy                           0.99     30003
   macro avg       0.99      0.99      0.99     30003
weighted avg       0.99      0.99      0.99     

In [34]:

if 'model.pth' not in os.listdir(op):
    torch.save(model.state_dict(), os.path.join(op, "model.pth"))
    print("Model saved")
else:
    print("model.pth confirmed in output directory.")

Model saved


In [37]:
np.save('train_loss.npy',train_loss)
np.save('test_loss.npy',test_loss)

In [39]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Model Size (Parameters)

What it means: The total count of individual weights and biases that your network updates during backpropagation. This dictates the static file size of the model when you save it to disk.

Why it matters: Larger models take up more memory and require more data to prevent overfitting.

In [44]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("--- MODEL SIZE PROFILE ---")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
# 1 float32 parameter = 4 bytes. 
print(f"Estimated Size on Disk: {(trainable_params * 4) / (1024 * 1024):.2f} MB")

--- MODEL SIZE PROFILE ---
Total Parameters: 1,851,258
Trainable Parameters: 1,851,258
Estimated Size on Disk: 7.06 MB


Inference Latency & Runtime Memory Footprint

Inference Latency: The exact time (in milliseconds) it takes for your model to process one single input stream through the network layers and output a prediction.

Runtime Memory Footprint: The peak amount of GPU Video RAM (VRAM) the model claims while performing that prediction pass.

Why it matters: High latency means your model cannot process real-time neuromorphic camera streams. High runtime memory prevents deploying the model to edge devices.

In [40]:
import time

def benchmark_cnn(model, device):
    model.eval()

    dummy_ip = torch.randn(1, 2, 30, 34, 34).to(device)

    torch.cuda.reset_peak_memory_stats(device)

    start_time = time.perf_counter()
    with torch.no_grad():
        for _ in range(100):
            _ = model(dummy_ip)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    end_time = time.perf_counter()

    avg_latency_ms = ((end_time - start_time) / 100) * 1000

    # Measure Runtime Memory Footprint
    if device.type == 'cuda':
        peak_memory_bytes = torch.cuda.max_memory_allocated(device)
        peak_memory_mb = peak_memory_bytes / (1024 * 1024)
    else:
        peak_memory_mb = 0.0

    print(f"Inference Latency: {avg_latency_ms:.4f} ms per event stream")
    print(f"Runtime Memory Footprint: {peak_memory_mb:.2f} MB")


In [41]:
benchmark_cnn(model, device)

Inference Latency: 0.6024 ms per event stream
Runtime Memory Footprint: 81.89 MB


Training Complexity & Resources

Training Time: The wall-clock time it takes to execute a single pass (epoch) over your training dataset.

Training Resources: The peak memory consumption forced onto your hardware during the forward and backward training loops.

Why it matters: Training requires calculating and holding gradients for every layer, meaning it consumes vastly more VRAM than simple inference.

In [42]:
for epoch in range(epochs):
    model.train()
    start_clock = time.time()
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_func(outputs, targets)
        loss.backward()
        optimizer.step()
    
    epoch_time = time.time() - start_clock
    
    if torch.cuda.is_available():
        peak_train_vram = torch.cuda.max_memory_allocated() / (1024 * 1024)
        print(f"Epoch {epoch+1:02d} | Duration: {epoch_time:.2f} sec | Peak Training VRAM: {peak_train_vram:.2f} MB")
    else:
        print(f"Epoch {epoch+1:02d} | Duration: {epoch_time:.2f} sec | Running on CPU")

Epoch 01 | Duration: 29.27 sec | Peak Training VRAM: 636.94 MB
Epoch 02 | Duration: 28.64 sec | Peak Training VRAM: 636.94 MB
Epoch 03 | Duration: 30.84 sec | Peak Training VRAM: 636.94 MB
Epoch 04 | Duration: 30.33 sec | Peak Training VRAM: 636.94 MB
Epoch 05 | Duration: 29.16 sec | Peak Training VRAM: 636.94 MB
Epoch 06 | Duration: 33.66 sec | Peak Training VRAM: 636.94 MB
Epoch 07 | Duration: 30.53 sec | Peak Training VRAM: 636.94 MB
Epoch 08 | Duration: 32.74 sec | Peak Training VRAM: 636.94 MB
Epoch 09 | Duration: 33.77 sec | Peak Training VRAM: 636.94 MB
Epoch 10 | Duration: 33.29 sec | Peak Training VRAM: 636.94 MB


Energy Consumption (Mathematical Estimation)

What it means: The total electrical energy (measured in Joules) required to execute one single inference. For a 3D CNN, you can compute it mathematically via MAC (Multiply-Accumulate) operations.In a 3D CNN, operations occur across every single pixel, regardless of whether it contains data or empty dark space. By industry standards on a standard 45nm CMOS chip, a single 32-bit MAC operation consumes roughly $4.6 \text{ pJ}$ (picojoules).

In [43]:
from thop import profile

# Build sample stream
dummy_stream = torch.randn(1, 2, 30, 34, 34).to(device)
macs, _ = profile(model, inputs=(dummy_stream,), verbose=False)

# Convert MACs to estimated Joules (4.6 pJ = 4.6 * 10^-12 Joules)
estimated_joules = macs * 4.6 * 1e-12

print("--- COGNITIVE WORKLOAD & ENERGY ---")
print(f"Total MAC Operations per Inference: {macs:,}")
print(f"Estimated Energy Consumption: {estimated_joules:.8f} Joules")

--- COGNITIVE WORKLOAD & ENERGY ---
Total MAC Operations per Inference: 94,501,248.0
Estimated Energy Consumption: 0.00043471 Joules
